# Chapter 1: Introduction to Python for Econometrics
### Book: Heiss & Brunner — *Using Python for Introductory Econometrics* (2024)

---

## What this chapter covers
1. Why Python for econometrics?
2. NumPy arrays — the numerical backbone
3. Pandas DataFrames — working with data tables
4. Loading real-world datasets and exploring them step by step
5. Creating meaningful visualisations
6. Descriptive statistics — manual *then* automated
7. Probability distributions and critical values
8. Confidence intervals — step by step
9. Hypothesis testing — step by step
10. Monte Carlo simulation — verifying theory with code

> **How to use this notebook:** Read every markdown cell *before* running the code.
> Each section explains **WHY** we do something, **WHEN** to use it,
> then walks through an **Example Question** solved step by step.

In [1]:
# ── SETUP ─────────────────────────────────────────────────────────────────
# Always run this cell first.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Fix working directory to this chapter's folder
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)

# Create output folder
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print('Working directory:', os.getcwd())
print('Output directory :', output_dir)
print('NumPy version    :', np.__version__)
print('Pandas version   :', pd.__version__)

Working directory: /Users/rubylambert/Library/CloudStorage/OneDrive-unine.ch/Econometrics/Using Python for Econometrics/Chapter_01_Introduction
Output directory : /Users/rubylambert/Library/CloudStorage/OneDrive-unine.ch/Econometrics/Using Python for Econometrics/Chapter_01_Introduction/output
NumPy version    : 2.3.5
Pandas version   : 2.3.3


---
## Section 1 — Why Python for Econometrics?

**Why not Excel or STATA?**
- Python is free, open-source, and used in academia AND industry
- It handles datasets of any size (Excel breaks at ~1 million rows)
- Everything is reproducible — your analysis is a script anyone can re-run
- Huge ecosystem: `statsmodels` for econometrics, `scipy` for statistics

**The libraries you will use constantly:**

| Library | What it does | When you use it |
|---------|-------------|----------------|
| `numpy` | Fast numerical arrays | Matrix operations, maths |
| `pandas` | Data tables (like Excel) | Loading, cleaning, summarising data |
| `matplotlib` | Plotting | Every time you visualise |
| `statsmodels` | Statistical models | OLS regression, hypothesis tests |
| `scipy.stats` | Probability distributions | CDFs, p-values, confidence intervals |

---
## Section 2 — NumPy Arrays: The Numerical Backbone

### WHY do we need arrays?
In econometrics, data comes as vectors (e.g. 526 wage observations) and matrices
(e.g. a 526×4 table of wages, education, experience, tenure).
Python's built-in `list` is too slow for numerical work.
NumPy arrays are implemented in C — fast and support element-wise maths.

### WHEN to use arrays vs lists?
- Use `list` when you have mixed types or just need a simple sequence
- Use `np.array` for *any* numerical computation

### Example Question
> You have 5 workers' hourly wages: $4.50, $7.20, $5.80, $9.10, $6.30.
> Compute the mean, deviations from the mean, and variance — step by step.

In [ ]:
# ── Step 1: Create the array ─────────────────────────────────────────────
wages = np.array([4.50, 7.20, 5.80, 9.10, 6.30])


Raw wages array: [4.5 7.2 5.8 9.1 6.3]
Type          : <class 'numpy.ndarray'>
Shape         : (5,)    <- (5,) means 1-D array with 5 elements
Data type     : float64   <- float64 = 64-bit floating point


In [ ]:

print('Raw wages array:', wages)
print('Type          :', type(wages))
print('Shape         :', wages.shape, '   <- (5,) means 1-D array with 5 elements')
print('Data type     :', wages.dtype, '  <- float64 = 64-bit floating point')

In [ ]:
# ── Step 2: Compute the mean MANUALLY so you understand the formula ───────
#
# Formula:  x_bar = (1/n) * sum(x_i)
#
n = len(wages)       # n = 5 observations


Sum of wages (total): 32.9
n (workers)         : 5
Mean (manual)       : 6.5800


In [ ]:
total = 0


In [ ]:
for w in wages: 
    total = total + w   # add up each wage one by one


In [ ]:

mean_manual = total / n
print(f'Sum of wages (total): {total}')
print(f'n (workers)         : {n}')
print(f'Mean (manual)       : {mean_manual:.4f}')

In [ ]:

# Now the NumPy shortcut — same result, one line:
mean_numpy = wages.mean()
print(f'Mean (numpy)        : {mean_numpy:.4f}')
print(f'Same result?        : {np.isclose(mean_manual, mean_numpy)}')

In [ ]:
# ── Step 3: Deviations from the mean ──────────────────────────────────────
#
# deviation_i = wage_i - mean
# This tells us: is each worker paid above or below average?
# NumPy broadcasts: subtracts the mean from EVERY element in one operation
#
deviations = wages - mean_numpy


In [ ]:

print('Worker | Wage    | Deviation  | Direction')
print('-' * 50)


In [ ]:
for i, (w, d) in enumerate(zip(wages, deviations)):
    direction = 'above average' if d > 0 else 'below average'
    print(f'  {i+1}    | ${w:.2f}  | {d:+.4f}    | {direction}')


In [ ]:

print(f'\nSum of all deviations: {deviations.sum():.12f}')
print('   -> Always equals zero. This is a mathematical property of the mean.')

In [ ]:
# ── Step 4: Variance and Standard Deviation ───────────────────────────────
#
# Variance:  s^2 = (1/(n-1)) * sum( (x_i - x_bar)^2 )
#
# WHY divide by n-1 instead of n?
# Because we ESTIMATED the mean from the same sample.
# This 'costs' one degree of freedom.
# Dividing by n-1 gives an UNBIASED estimate of population variance.
# This is called the 'sample variance'.
#
squared_devs = deviations ** 2
print('Squared deviations:', squared_devs.round(4))

variance_manual = squared_devs.sum() / (n - 1)
std_manual      = np.sqrt(variance_manual)

print(f'\nVariance (manual):    {variance_manual:.4f}')
print(f'Std Dev  (manual):    {std_manual:.4f}')

# NumPy shortcuts (ddof=1 means divide by n-1)
print(f'Variance (numpy):     {wages.var(ddof=1):.4f}')
print(f'Std Dev  (numpy):     {wages.std(ddof=1):.4f}')

print('\nInterpretation:')
print(f'  Mean wage     = ${mean_numpy:.2f}/hour')
print(f'  Std deviation = ${std_manual:.2f}/hour')
print(f'  -> The typical wage is within ${std_manual:.2f} of the mean.')
print(f'  -> Wages range from ${wages.min():.2f} to ${wages.max():.2f}.')

---
## Section 3 — Pandas DataFrames: Working with Real Data Tables

### WHY Pandas?
Real datasets have multiple columns, variable names, missing values, and mixed types.
NumPy arrays cannot handle all of that.
Pandas gives us a DataFrame — a named table where:
- Each **row** = one observation (one person, one firm, one time period)
- Each **column** = one variable
- Columns have **names** so you never confuse column 3 with column 7

### WHEN to use a DataFrame?
Whenever your data has more than one variable. In econometrics, that's basically always.

### Example Question
> Build a small DataFrame of workers, then:
> (a) Show what it looks like
> (b) Select only female workers
> (c) Create a new log-wage column
> (d) Show the data before and after each transformation

In [ ]:
# ── Build a DataFrame from scratch ───────────────────────────────────────
data = {
    'name'  : ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'wage'  : [4.50,    7.20,  5.80,    9.10,    6.30],
    'educ'  : [10,      14,    12,      16,      12  ],
    'female': [1,       0,     1,       0,       1   ],
}

df_small = pd.DataFrame(data)

print('Our DataFrame (one row = one worker):')
print(df_small)
print()
print('Shape :', df_small.shape, '  <- (rows, columns)')
print('Column names:', list(df_small.columns))
print('Index (row labels):', list(df_small.index))

In [ ]:
# ── Selecting columns ─────────────────────────────────────────────────────
# A single column returns a Series (1-D labelled array)
wage_col = df_small['wage']
print('wage column (type = Series):')
print(wage_col)
print('Type:', type(wage_col))

print()

# Multiple columns return a DataFrame
subset = df_small[['name', 'wage', 'educ']]
print('Subset with 3 columns (type = DataFrame):')
print(subset)

In [ ]:
# ── Filtering rows: select female workers only ────────────────────────────
# Step 1: Create a boolean mask (True/False for each row)
is_female = df_small['female'] == 1

print('Boolean mask (is female?):')
print(is_female.to_frame())

# Step 2: Use the mask to keep only rows where mask=True
female_workers = df_small[is_female]

print('\nFemale workers only:')
print(female_workers)

print()
print(f'Mean wage (ALL workers)    : ${df_small["wage"].mean():.2f}')
print(f'Mean wage (FEMALE workers) : ${female_workers["wage"].mean():.2f}')
print(f'Mean wage (MALE workers)   : ${df_small[df_small["female"]==0]["wage"].mean():.2f}')
gap = df_small[df_small['female']==0]['wage'].mean() - female_workers['wage'].mean()
print(f'Raw gender wage gap        : ${gap:.2f}/hour')

In [ ]:
# ── Creating new columns: log-wage transformation ─────────────────────────
# WHY log(wage)?
# Wages are right-skewed. Log makes the distribution more symmetric.
# In a log-level regression: coeff = % change in wage per unit change in x

print('BEFORE adding new column:')
print(df_small[['name','wage']].to_string(index=False))

# Add log_wage
df_small['log_wage'] = np.log(df_small['wage'])

# Add standardised wage: (x - mean) / std -> mean=0, std=1
df_small['wage_z'] = (df_small['wage'] - df_small['wage'].mean()) / df_small['wage'].std()

print('\nAFTER adding log_wage and wage_z:')
print(df_small[['name','wage','log_wage','wage_z']].round(3).to_string(index=False))

print('\nInterpretation:')
print('  log_wage: natural log of wage')
print('  wage_z : standardised wage (number of std devs from the mean)')
print('  wage_z > 0 -> earns above average; wage_z < 0 -> earns below average')

---
## Section 4 — Loading Real Data: The Wooldridge Datasets

### WHY use Wooldridge datasets?
These are real economic datasets from Wooldridge's *Introductory Econometrics* textbook.
They are well-documented, widely used in teaching, and cover diverse topics.

### The `wage1` dataset
- **Source**: 1976 Current Population Survey (USA)
- **526 observations** of working individuals
- **Key variables**: `wage` (hourly wage), `educ` (years of education),
  `exper` (years of experience), `tenure` (years with current employer),
  `female` (=1 if female), `married` (=1 if married)

### Example Question
> Load the `wage1` dataset. Fully explore it:
> What is its size? What variables exist? What does a typical row look like?
> Are there missing values? What is the distribution of wages?

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────
import wooldridge as woo

df = woo.dataWoo('wage1')

print('=' * 55)
print('STEP 1: Size of the dataset')
print('=' * 55)
rows, cols = df.shape
print(f'  Rows    (observations): {rows}')
print(f'  Columns (variables)   : {cols}')

In [ ]:
# ── STEP 2: What does the raw data look like? ─────────────────────────────
print('First 5 rows of the raw dataset:')
print('-' * 70)
print(df.head().to_string())

print('\nLast 5 rows:')
print('-' * 70)
print(df.tail().to_string())

In [ ]:
# ── STEP 3: What variables do we have? ───────────────────────────────────
print('All variable names:')
print(list(df.columns))

print('\nData types:')
print(df.dtypes.to_string())

In [ ]:
# ── STEP 4: Are there missing values? ────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
if missing.any():
    print(missing[missing > 0])
else:
    print('  -> No missing values. Dataset is complete.')

# STEP 5: Summary statistics for key variables
key_vars = ['wage', 'educ', 'exper', 'tenure']
summary = df[key_vars].describe().T
summary.columns = ['n','mean','std','min','25%','median','75%','max']
print('\nSummary statistics:')
print(summary.round(2).to_string())

# Save
summary.round(2).to_csv(os.path.join(output_dir, 'table_ch01_summary_stats.csv'))
print('\n-> Table saved to output/table_ch01_summary_stats.csv')

In [ ]:
# ── STEP 6: Interpret the numbers ────────────────────────────────────────
print('INTERPRETATION OF SUMMARY STATISTICS:')
print()
mean_w  = df['wage'].mean()
med_w   = df['wage'].median()
std_w   = df['wage'].std()
print(f'  Wage:')
print(f'    Mean    = ${mean_w:.2f}/hour')
print(f'    Median  = ${med_w:.2f}/hour')
print(f'    Mean > Median -> RIGHT-SKEWED (a few very high earners pull the mean up)')
print(f'    Std Dev = ${std_w:.2f}/hour   (typical spread around the mean)')
print(f'    Range   = ${df["wage"].min():.2f} to ${df["wage"].max():.2f}/hour')
print()
print(f'  Education:')
print(f'    Mean = {df["educ"].mean():.1f} years')
print(f'    Mode = {df["educ"].mode()[0]} years  (most common = high school diploma)')
print()
n_f = df['female'].sum()
pct_f = 100 * n_f / len(df)
print(f'  Gender: {int(n_f)} female ({pct_f:.1f}%), {int(len(df)-n_f)} male ({100-pct_f:.1f}%)')
gap = df[df['female']==0]['wage'].mean() - df[df['female']==1]['wage'].mean()
print(f'  Raw gender gap: ${gap:.2f}/hour  (unadjusted for education/experience)')

---
## Section 5 — Data Visualisation

### WHY visualise before modelling?
> *'A picture is worth a thousand regressions.'*

Looking at your data before fitting models helps you:
- Spot skewness → decide whether to use log transformation
- See if relationships look linear → choose model form
- Find outliers → check if they will distort estimates

### WHEN to use which plot?
| Plot | Use it when... |
|------|---------------|
| Histogram | Understand distribution of one variable |
| Scatter | Explore relationship between two variables |
| Box plot | Compare distributions across groups |
| Line plot | Data over time |

### Example Question
> Plot the distribution of wages and compare it to log(wages).
> Then plot wages against education. What do these plots tell you before
> you run a single regression?

In [ ]:
# ── Histogram: Wage vs Log(Wage) distribution ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: wage
axes[0].hist(df['wage'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['wage'].mean(),   color='red',    linestyle='--', lw=2,
    label=f'Mean = ${df["wage"].mean():.2f}')
axes[0].axvline(df['wage'].median(), color='orange', linestyle='--', lw=2,
    label=f'Median = ${df["wage"].median():.2f}')
axes[0].set_title('Distribution of Hourly Wages', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hourly Wage ($/hour)'); axes[0].set_ylabel('Number of Workers')
axes[0].legend()
axes[0].text(0.97, 0.95,
    f'Skewness = {df["wage"].skew():.2f}\n(positive = right-skewed)',
    transform=axes[0].transAxes, ha='right', va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Right: log(wage)
lwage = np.log(df['wage'])
axes[1].hist(lwage, bins=30, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(lwage.mean(), color='red', linestyle='--', lw=2,
    label=f'Mean = {lwage.mean():.2f}')
axes[1].set_title('Distribution of log(Wages)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Hourly Wage)'); axes[1].set_ylabel('Number of Workers')
axes[1].legend()
axes[1].text(0.97, 0.95,
    f'Skewness = {lwage.skew():.2f}\n(closer to normal)',
    transform=axes[1].transAxes, ha='right', va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('Why economists use log(wage) in regressions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_wage_hist.png'), dpi=150, bbox_inches='tight')
plt.show()

print('KEY INSIGHT:')
print('  Wages are RIGHT-SKEWED: mean > median, long tail to the right.')
print('  Log(wages) is much more symmetric -> closer to normal distribution.')
print('  This is WHY economists regress log(wage) rather than wage.')
print('  Coefficient interpretation: 1-unit increase in x -> beta*100 % change in wage.')

In [ ]:
# ── Scatter plot: Wage vs Education ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = df['female'].map({0: 'steelblue', 1: 'crimson'})

# Left: wage vs educ
axes[0].scatter(df['educ'], df['wage'], c=colors, alpha=0.35, s=18)
z = np.polyfit(df['educ'], df['wage'], 1)
x_line = np.linspace(df['educ'].min(), df['educ'].max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), 'k-', lw=2,
    label=f'Trend (slope={z[0]:.2f})')
axes[0].set_title('Wage vs Years of Education', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years of Education'); axes[0].set_ylabel('Hourly Wage ($/hour)')
axes[0].legend()

# Right: log(wage) vs educ  -- this is what OLS will model
axes[1].scatter(df['educ'], np.log(df['wage']), c=colors, alpha=0.35, s=18)
z2 = np.polyfit(df['educ'], np.log(df['wage']), 1)
axes[1].plot(x_line, np.poly1d(z2)(x_line), 'k-', lw=2,
    label=f'Trend (slope={z2[0]:.3f})')
axes[1].set_title('log(Wage) vs Years of Education', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Years of Education'); axes[1].set_ylabel('log(Hourly Wage)')
axes[1].legend()

from matplotlib.lines import Line2D
legend_els = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', ms=8, label='Male'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='crimson',   ms=8, label='Female')]
for ax in axes: ax.legend(handles=legend_els + [ax.get_lines()[-1]], fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_wage_vs_educ.png'), dpi=150, bbox_inches='tight')
plt.show()

r1 = df['wage'].corr(df['educ'])
r2 = np.log(df['wage']).corr(df['educ'])
print(f'Correlation (wage,  educ) = {r1:.3f}')
print(f'Correlation (lwage, educ) = {r2:.3f}')
print(f'Slope (wage ~ educ)    : each extra year of education -> +${z[0]:.2f}/hour on average')
print(f'Slope (lwage ~ educ)   : each extra year of education -> +{z2[0]*100:.1f}% wage on average')

In [ ]:
# ── Box plot: Wages by education group and gender ─────────────────────────
df['educ_group'] = pd.cut(df['educ'],
    bins=[0, 11, 12, 15, 18],
    labels=['<12 yrs\n(<HS)', '12 yrs\n(HS)', '13-15 yrs\n(Some Coll)', '16+ yrs\n(College+)'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot by education group
groups = [df[df['educ_group']==g]['wage'].values for g in df['educ_group'].cat.categories]
bp = axes[0].boxplot(groups, patch_artist=True, medianprops=dict(color='red', lw=2))
for patch, col in zip(bp['boxes'], ['#d4e6f1','#a9cce3','#5dade2','#1a5276']):
    patch.set_facecolor(col)
axes[0].set_xticklabels(df['educ_group'].cat.categories, fontsize=8)
axes[0].set_title('Wage Distribution by Education Level', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Hourly Wage ($/hour)')

# Box plot by gender
g_groups = [df[df['female']==0]['wage'].values, df[df['female']==1]['wage'].values]
bp2 = axes[1].boxplot(g_groups, patch_artist=True, medianprops=dict(color='red', lw=2))
bp2['boxes'][0].set_facecolor('steelblue')
bp2['boxes'][1].set_facecolor('crimson')
axes[1].set_xticklabels(['Male', 'Female'])
axes[1].set_title('Wage Distribution by Gender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Hourly Wage ($/hour)')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Mean wages by education group:')
print(df.groupby('educ_group', observed=True)['wage'].mean().round(2).to_string())
print()
mw = df[df['female']==0]['wage'].mean()
fw = df[df['female']==1]['wage'].mean()
print(f'Mean wage (Male)  : ${mw:.2f}/hour')
print(f'Mean wage (Female): ${fw:.2f}/hour')
print(f'Raw gap           : ${mw-fw:.2f}/hour (unadjusted -- Chapter 3 controls for education & experience)')

---
## Section 6 — Probability Distributions

### WHY do we need them?
Every test statistic we compute (t, F, chi-squared) follows a known probability
distribution *under the null hypothesis*. We need these to:
- Compute **p-values**: P(seeing a result this extreme | H0 is true)
- Find **critical values**: the cutoff for 'statistically significant'
- Build **confidence intervals**

### The four distributions you'll see every chapter

| Distribution | Symbol | Appears when... |
|-------------|--------|-----------------|
| Normal | N(mu, sigma^2) | Large-sample test statistics, error terms |
| Student's t | t(df) | t-tests for regression coefficients |
| Chi-squared | chi2(df) | Heteroscedasticity tests, specification tests |
| F | F(q, n-k-1) | Joint significance tests (F-tests) |

### Key relationship
If Z ~ N(0,1) then Z^2 ~ chi2(1). The t-distribution is a ratio of Normal/sqrt(chi2).
The F-distribution is a ratio of two chi-squareds. They are all connected!

In [ ]:
# ── Visualise all four distributions ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Normal
x = np.linspace(-5, 5, 500)
for mu, sigma, label in [(0,1,'N(0,1)'), (0,2,'N(0,4)'), (1,1,'N(1,1)')]:
    axes[0,0].plot(x, stats.norm.pdf(x, mu, sigma), lw=2, label=label)
axes[0,0].set_title('Normal Distribution', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('Density')
axes[0,0].legend()
axes[0,0].text(0.05, 0.9, '68% of N(0,1)\nwithin +-1 std dev\n95% within +-1.96',
    transform=axes[0,0].transAxes, fontsize=8,
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 2. t-distribution
for df_val, label in [(2,'t(2)'),(5,'t(5)'),(30,'t(30)'),(1000,'t(1000) ~ N(0,1)')]:
    axes[0,1].plot(x, stats.t.pdf(x, df_val), lw=2, label=label)
axes[0,1].set_title('t-Distribution (fatter tails than Normal)', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('t'); axes[0,1].set_ylabel('Density')
axes[0,1].legend(fontsize=8)
axes[0,1].text(0.05, 0.9, 'As df increases,\nt approaches Normal.\nUse t for small n.',
    transform=axes[0,1].transAxes, fontsize=8,
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 3. Chi-squared
x_pos = np.linspace(0.01, 25, 500)
for df_val, label in [(1,'chi2(1)'),(3,'chi2(3)'),(5,'chi2(5)'),(10,'chi2(10)')]:
    axes[1,0].plot(x_pos, stats.chi2.pdf(x_pos, df_val), lw=2, label=label)
axes[1,0].set_title('Chi-Squared Distribution', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('Density')
axes[1,0].legend()
axes[1,0].text(0.55, 0.9, 'Always positive\nRight-skewed\nMean = df',
    transform=axes[1,0].transAxes, fontsize=8,
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 4. F-distribution
x_f = np.linspace(0.01, 6, 500)
for d1, d2, label in [(1,50,'F(1,50)'),(2,50,'F(2,50)'),(5,50,'F(5,50)'),(10,50,'F(10,50)')]:
    axes[1,1].plot(x_f, stats.f.pdf(x_f, d1, d2), lw=2, label=label)
axes[1,1].set_title('F-Distribution (for joint tests)', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('F'); axes[1,1].set_ylabel('Density')
axes[1,1].legend()
axes[1,1].text(0.55, 0.9, 'Always positive\nUsed for F-tests\nin regression',
    transform=axes[1,1].transAxes, fontsize=8,
    bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('The Four Key Distributions in Econometrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Critical values and p-values in practice ─────────────────────────────
print('Critical values for t-distribution (two-tailed, alpha=0.05):')
print('n     | df    | t_crit')
print('-' * 30)
for n in [10, 30, 100, 500, 10000]:
    cv = stats.t.ppf(0.975, df=n-1)
    print(f'n={n:<5} | df={n-1:<4} | {cv:.4f}')
print('n=inf  | df=inf | 1.9600  (standard normal z-critical)')

print()
print('How to compute a p-value (two-tailed):')
t_example = 2.35
n_example  = 100
p = 2 * stats.t.sf(abs(t_example), df=n_example-1)
print(f'  t = {t_example}, n = {n_example}')
print(f'  p-value = 2 * P(T > |{t_example}|) with {n_example-1} df = {p:.4f}')
print(f'  At alpha=0.05: {"REJECT H0" if p<0.05 else "FAIL TO REJECT H0"}')
print(f'  Meaning: if H0 were true, we would observe a t-stat this extreme')
print(f'  only {p*100:.1f}% of the time by chance.')

---
## Section 7 — Confidence Intervals: Step by Step

### WHY confidence intervals?
A point estimate (e.g. mean wage = $5.90) is based on a *sample*.
A different sample would give a different number.
A confidence interval quantifies that uncertainty.

> **Correct interpretation of a 95% CI:**
> If we repeated the sampling procedure many times, 95% of the intervals
> we construct would contain the true population mean.
> (The true mean is fixed — the interval is what's random!)

### Formula
$$\bar{x} \pm t_{\alpha/2,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

### Example Question
> Using `wage1`, build a 95% CI for the mean hourly wage.
> Does the interval include $5.00?

In [ ]:
# ── Step-by-step: 95% CI for mean hourly wage ────────────────────────────
print('STEP-BY-STEP: 95% Confidence Interval for Mean Hourly Wage')
print('=' * 60)

# Step 1: Sample statistics
x_bar = df['wage'].mean()
s     = df['wage'].std(ddof=1)   # ddof=1 -> divide by n-1
n     = len(df)

print('Step 1 -- Sample statistics:')
print(f'  x_bar (sample mean)    = {x_bar:.4f}')
print(f'  s     (sample std dev) = {s:.4f}  [using ddof=1]')
print(f'  n     (observations)   = {n}')

# Step 2: t critical value
alpha  = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)   # 97.5th percentile

print(f'Step 2 -- t critical value:')
print(f'  alpha={alpha} -> need t_{{alpha/2, n-1}} = t_{{0.025, {n-1}}}')
print(f'  t_crit = {t_crit:.4f}')

# Step 3: Standard error
SE = s / np.sqrt(n)
print(f'Step 3 -- Standard error of the mean:')
print(f'  SE = s / sqrt(n) = {s:.4f} / sqrt({n}) = {SE:.4f}')

# Step 4: Margin of error and CI
ME = t_crit * SE
CI_lower = x_bar - ME
CI_upper = x_bar + ME

print(f'Step 4 -- Margin of error:')
print(f'  ME = t_crit * SE = {t_crit:.4f} * {SE:.4f} = {ME:.4f}')

print(f'Step 5 -- 95% Confidence Interval:')
print(f'  [{x_bar:.4f} - {ME:.4f},  {x_bar:.4f} + {ME:.4f}]')
print(f'  = [{CI_lower:.2f},  {CI_upper:.2f}]')

print('INTERPRETATION:')
print(f'  We are 95% confident the true mean hourly wage (1976 USA)')
print(f'  lies between ${CI_lower:.2f} and ${CI_upper:.2f}.')
in_ci = CI_lower <= 5.00 <= CI_upper
print(f'  Does $5.00 fall inside the CI? {in_ci}')
print(f'  -> {"CANNOT reject H0: mu=5 at the 5% level" if in_ci else "REJECT H0: mu=5 at the 5% level"}')

---
## Section 8 — Hypothesis Testing: Step by Step

### WHY hypothesis testing?
We have a sample. We want to make a claim about the **population**.
Hypothesis testing gives us a rigorous procedure to decide whether the
data provides enough evidence against the null hypothesis.

### The 5-Step Procedure (memorise this!)
1. **State H0 and H1** — be precise
2. **Choose alpha** — typically 0.05 or 0.01
3. **Compute the test statistic** — how many standard errors from H0?
4. **Find the p-value** — P(result this extreme | H0 true)
5. **Make a decision and interpret** economically

### Example Question
> Test whether the average hourly wage is significantly greater than $5.00.
> Use a one-sided test at the 5% significance level.
> Show all 5 steps and plot the rejection region.

In [ ]:
# ── One-sample t-test: H0: mu=5 vs H1: mu>5 ─────────────────────────────
print('HYPOTHESIS TEST: Is the mean hourly wage significantly above $5.00?')
print('=' * 60)

print('Step 1: Hypotheses')
print('  H0: mu = $5.00/hour   (null: average wage equals $5)')
print('  H1: mu > $5.00/hour   (alternative: average wage exceeds $5)')
print('  One-sided test (right-tail)')

mu_0  = 5.00
x_bar = df['wage'].mean()
s     = df['wage'].std(ddof=1)
n     = len(df)

print('Step 2: Significance level')
print('  alpha = 0.05 (5%)')

t_stat = (x_bar - mu_0) / (s / np.sqrt(n))
print('Step 3: t-statistic')
print(f'  t = (x_bar - mu_0) / (s/sqrt(n))')
print(f'    = ({x_bar:.4f} - {mu_0}) / ({s:.4f}/sqrt({n}))')
print(f'    = {x_bar-mu_0:.4f} / {s/np.sqrt(n):.4f}')
print(f'    = {t_stat:.4f}')

p_val       = stats.t.sf(t_stat, df=n-1)           # P(T > t_stat)
t_crit_one  = stats.t.ppf(0.95,  df=n-1)           # one-sided 5%
print('Step 4: p-value')
print(f'  p-value = P(T > {t_stat:.4f}) with {n-1} df = {p_val:.6f}')
print(f'  Critical value: t_{{0.05, {n-1}}} = {t_crit_one:.4f}')

print('Step 5: Decision')
print(f'  t_stat ({t_stat:.4f}) {">">"<"[int(t_stat<t_crit_one)]} t_crit ({t_crit_one:.4f})')
print(f'  p-value ({p_val:.6f}) {"<" if p_val<0.05 else ">="} 0.05')
if p_val < 0.05:
    print('  -> REJECT H0 at the 5% level.')
    print('  -> CONCLUSION: The mean wage is statistically significantly above $5/hour.')
else:
    print('  -> FAIL TO REJECT H0.')

# Plot the rejection region
fig, ax = plt.subplots(figsize=(10, 4))
x_t = np.linspace(-4, max(t_stat+1, 7), 500)
ax.plot(x_t, stats.t.pdf(x_t, df=n-1), 'b-', lw=2, label=f't({n-1}) under H0')
x_rej = np.linspace(t_crit_one, x_t.max(), 200)
ax.fill_between(x_rej, stats.t.pdf(x_rej, df=n-1), color='red', alpha=0.4,
    label=f'Rejection region (5%)')
ax.axvline(t_stat,     color='green', lw=2.5, linestyle='--', label=f'Our t = {t_stat:.2f}')
ax.axvline(t_crit_one, color='red',   lw=1.5, linestyle=':',  label=f't_crit = {t_crit_one:.2f}')
ax.set_title('Hypothesis Test: H0: mu=5 vs H1: mu>5', fontsize=13, fontweight='bold')
ax.set_xlabel('t-statistic'); ax.set_ylabel('Density')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_hypothesis_test.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9 — Monte Carlo Simulation

### WHY simulate?
Monte Carlo lets us *verify theoretical results by experiment*.
Instead of just trusting a mathematical proof, we:
1. Simulate a world where we know the truth
2. Run the experiment thousands of times
3. Check that the theory matches the simulation

### Two results to verify
1. **Law of Large Numbers (LLN):** as n increases, the sample mean converges to the true mean
2. **Central Limit Theorem (CLT):** sample means are approximately normal,
   even if the underlying distribution is NOT normal

These two results are WHY OLS inference works in large samples!

### Example Question
> Simulate wages from a right-skewed distribution (like real wages).
> (a) Show that the sample mean converges to the true mean as n grows (LLN).
> (b) Show that the distribution of sample means approaches normal (CLT).

In [ ]:
# ── LAW OF LARGE NUMBERS ─────────────────────────────────────────────────
np.random.seed(42)

# Simulate from Exponential(mean=5) -- RIGHT-SKEWED, like real wages
true_mean = 5.0
n_max = 2000
simulated_wages = np.random.exponential(scale=true_mean, size=n_max)

print('First 10 simulated wages:', simulated_wages[:10].round(2))
print(f'True mean: {true_mean}')

# Running (cumulative) mean after each additional observation
running_mean = np.cumsum(simulated_wages) / np.arange(1, n_max + 1)

print(f'\nRunning mean after:')
for k in [1, 5, 10, 50, 100, 500, 1000, 2000]:
    print(f'  n={k:5d}: mean = {running_mean[k-1]:.4f}  (error = {abs(running_mean[k-1]-true_mean):.4f})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(range(1, n_max+1), running_mean, color='steelblue', lw=1)
axes[0].axhline(true_mean, color='red', lw=2, linestyle='--', label=f'True mean = {true_mean}')
axes[0].set_xscale('log')
axes[0].set_title('Law of Large Numbers:\nSample Mean Converges to True Mean', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sample size n (log scale)'); axes[0].set_ylabel('Sample Mean')
axes[0].legend()

axes[1].hist(simulated_wages, bins=50, color='darkorange', edgecolor='white', alpha=0.85, density=True)
x_exp = np.linspace(0, 30, 300)
axes[1].plot(x_exp, stats.expon.pdf(x_exp, scale=true_mean), 'r-', lw=2, label='True density')
axes[1].axvline(true_mean, color='green', linestyle='--', lw=2, label=f'True mean={true_mean}')
axes[1].set_title('Underlying Distribution\n(RIGHT-SKEWED -- not normal!)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Simulated Wage'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_LLN.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CENTRAL LIMIT THEOREM ─────────────────────────────────────────────────
np.random.seed(42)
n_sims = 5000   # number of repeated experiments

print('We draw samples from the SKEWED exponential distribution.')
print('For each sample size n, we compute 5000 sample means.')
print('The CLT says: the distribution of sample means -> Normal, as n grows.')
print()

sample_sizes = [5, 30, 100, 500]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, n in zip(axes, sample_sizes):
    # Draw n_sims samples of size n, compute mean of each
    sample_means = np.array([
        np.random.exponential(scale=true_mean, size=n).mean()
        for _ in range(n_sims)
    ])
    # CLT theoretical distribution: Normal(mu, sigma^2/n)
    # For exponential: sigma = mean = 5
    theo_std = true_mean / np.sqrt(n)

    ax.hist(sample_means, bins=50, density=True, color='steelblue',
        edgecolor='white', alpha=0.8)
    x_range = np.linspace(sample_means.min(), sample_means.max(), 300)
    ax.plot(x_range, stats.norm.pdf(x_range, true_mean, theo_std), 'r-', lw=2.5)
    ax.set_title(f'n = {n}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Mean')
    if n == 5: ax.set_ylabel('Density')
    ax.text(0.05, 0.93,
        f'Sim std={sample_means.std():.3f}\nCLT std={theo_std:.3f}',
        transform=ax.transAxes, fontsize=7, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('Central Limit Theorem\nSample means approach Normal even from a skewed distribution',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch01_CLT.png'), dpi=150, bbox_inches='tight')
plt.show()

print('KEY RESULT:')
print('  Even though wages are RIGHT-SKEWED,')
print('  sample means are APPROXIMATELY NORMAL for n >= 30.')
print()
print('WHY THIS MATTERS:')
print('  OLS t-tests and F-tests rely on normality of the ESTIMATOR,')
print('  not the error term. With large n, this holds via the CLT.')
print('  -> This justifies normal-based inference even with non-normal residuals.')

---
## Chapter 1 Summary

### What you learned

| Concept | Key formula | Python |
|---------|------------|--------|
| Sample mean | x_bar = (1/n) sum(x_i) | `df['col'].mean()` |
| Sample std | s = sqrt(var(x)) | `df['col'].std(ddof=1)` |
| Standard error | SE = s/sqrt(n) | manual |
| 95% CI | x_bar +- t_{0.025,n-1} * SE | `stats.t.interval(0.95,...)` |
| t-statistic | t = (x_bar - mu_0) / SE | manual |
| p-value (two-sided) | 2*P(T>|t|) | `stats.t.sf(abs(t),df)*2` |

### Key insights from `wage1`
- Wages are **right-skewed** → use log(wage) in regressions (Chapter 2)
- Education has a **positive correlation** with wages (r = 0.41)
- There is a **raw gender gap** of ~$2.50/hour — but this is unadjusted
  (Chapter 3 will control for education and experience)
- Mean wage is **significantly above $5** (p < 0.001)

### Practice Questions
1. Compute a 99% confidence interval for mean years of education. Does it include 12?
2. Test whether the proportion of female workers equals 0.5 (two-sided, alpha=0.05).
3. Create a histogram of `exper`. Is it skewed? Would log(exper+1) be more symmetric?
4. What is the correlation between `exper` and `tenure`? Why might this be positive?
5. Using a two-sided test, is the average education significantly different from 13 years?

### What comes next
**Chapter 2:** Simple regression — formally model wage vs education
**Chapter 3:** Multiple regression — control for several variables simultaneously